# Colab Qwen-VL Benchmark (Bailian International)

This notebook runs the benchmark logic with Qwen-VL API using the API key `ALI_INTERNATIONAL_KEY`, and writes outputs to Google Drive.

- API source: Alibaba Cloud Bailian (International) Compatible API
- Qwen model fallback list and generation params: qwen-vl-max / qwen-vl-plus
- Output path: Google Drive

In [ ]:
!pip -q install openai pandas numpy tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import csv
import json
import random
from pathlib import Path
from datetime import datetime

from openai import OpenAI

# ====== Config ======
API_ENV_NAME = 'ALI_INTERNATIONAL_KEY'

# Input CSV
BENCHMARK_QA_CSV = os.getenv(
    'BENCHMARK_QA_CSV',
    '/content/drive/MyDrive/CCD_VQA/VRU/vid_list/multi_version_data/generated_options_5opts_20260327_085451.csv'
)

# Output directory on Drive
DRIVE_OUTPUT_DIR = os.getenv(
    'DRIVE_OUTPUT_DIR',
    '/content/drive/MyDrive/CCD_VQA/benchmark_result'
)

MODEL_NAME = 'qwen-vl-max'

# Qwen-VL fallback models (using OpenAI compatible model names for Bailian)
MODEL_CANDIDATES = [
    'qwen-vl-max-latest',
    'qwen-vl-max',
    'qwen-vl-plus'
]

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# load API key from Colab Secrets or env var
try:
    from google.colab import userdata
    api_key = userdata.get(API_ENV_NAME)
except (ImportError, getattr(userdata, 'SecretNotFoundError', Exception)):
    api_key = os.getenv(API_ENV_NAME)

if not api_key:
    raise RuntimeError(f'Missing API key in Colab Secrets or env var: {API_ENV_NAME}. Please make sure you added "{API_ENV_NAME}" in the Secrets tab and toggled notebook access.')

# Initialize OpenAI client pointing to Alibaba Bailian International
client = OpenAI(
    api_key=api_key,
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
)

print('API env name:', API_ENV_NAME)
print('CSV:', BENCHMARK_QA_CSV)
print('Output dir:', DRIVE_OUTPUT_DIR)

In [ ]:
def load_questions_from_csv(csv_path):
    questions_per_video = []
    encodings_to_try = ['utf-8-sig', 'utf-8', 'gb18030', 'cp1252', 'latin1']
    last_error = None

    for encoding in encodings_to_try:
        try:
            with open(csv_path, encoding=encoding) as f:
                reader = csv.DictReader(f)
                fieldnames = reader.fieldnames or []

                if 'video_id' in fieldnames and 'question' in fieldnames and 'correct_answer' in fieldnames:
                    video_to_qas = {}
                    for row in reader:
                        video_number = (row.get('video_id') or '').strip()
                        q_text = (row.get('question') or '').strip()
                        correct_answer = (row.get('correct_answer') or '').strip()

                        if not video_number or not q_text or not correct_answer:
                            continue

                        options = [correct_answer]
                        option_idx = 1
                        while True:
                            option_key = f'option_{option_idx}'
                            if option_key not in row:
                                break
                            wrong_opt = (row.get(option_key) or '').strip()
                            if wrong_opt:
                                options.append(wrong_opt)
                            option_idx += 1

                        original_correct_idx = 0
                        idx_list = list(range(len(options)))
                        random.shuffle(idx_list)
                        shuffled_options = [options[j] for j in idx_list]
                        correct_index = idx_list.index(original_correct_idx) + 1

                        qa = {
                            'question': q_text,
                            'options': shuffled_options,
                            'correct_index': correct_index,
                            'question_id': int(row.get('question_id') or 0),
                        }
                        video_to_qas.setdefault(video_number, []).append(qa)

                    for video_number in sorted(video_to_qas.keys()):
                        qas = sorted(video_to_qas[video_number], key=lambda x: x.get('question_id', 0))
                        for qa in qas:
                            qa.pop('question_id', None)
                        questions_per_video.append({'video_number': video_number, 'qas': qas})
                else:
                    for row in reader:
                        video_number = row['video_number']
                        qas = []
                        for i in range(1, 7):
                            q_text = row.get(f'q{i}_text', None)
                            if not q_text:
                                continue

                            options = []
                            answer = row.get(f'q{i}_ans_correct', None)
                            if answer:
                                options.append(answer)

                            for k in range(1, 6):
                                wrong_key = f'q{i}_ans_wrong{k}'
                                wrong_opt = row.get(wrong_key, None)
                                if wrong_opt and wrong_opt.strip():
                                    options.append(wrong_opt)

                            if not options:
                                continue

                            original_correct_idx = 0
                            idx_list = list(range(len(options)))
                            random.shuffle(idx_list)
                            shuffled_options = [options[j] for j in idx_list]
                            correct_index = idx_list.index(original_correct_idx) + 1

                            qas.append({
                                'question': q_text,
                                'options': shuffled_options,
                                'correct_index': correct_index,
                            })

                        questions_per_video.append({'video_number': video_number, 'qas': qas})

            if encoding != 'utf-8-sig':
                print(f'CSV encoding fallback success: {encoding}')
            break
        except UnicodeDecodeError as e:
            last_error = e
            questions_per_video = []
            continue

    if not questions_per_video and last_error is not None:
        raise UnicodeDecodeError(
            last_error.encoding,
            last_error.object,
            last_error.start,
            last_error.end,
            f'CSV decode failed. tried: {encodings_to_try}, error: {last_error.reason}',
        )

    return questions_per_video

def build_single_question_prompt(qa_index, qa):
    q_text = qa['question'] if qa['question'] else ''
    num_options = len(qa.get('options', []))
    if num_options <= 0:
        num_options = 4

    option_numbers = ', '.join(str(i) for i in range(1, num_options + 1))
    prompt = f'Answer this multiple choice question. Respond with ONLY one number from: {option_numbers}. No explanation.\n\n'
    prompt += f'Question {qa_index}: {q_text}\n'
    for opt_idx, opt in enumerate(qa['options'], 1):
        prompt += f'  {opt_idx}. {opt}\n'
    prompt += '\nYour answer (only the number): '
    return prompt

def parse_choices(response, num_options=4, expected_count=1):
    if num_options <= 0:
        num_options = 4
    if expected_count <= 0:
        expected_count = 1

    max_opt = str(num_options)
    opt_char_class = f'[1-{max_opt}]'
    choices = []

    pattern1 = rf'(\d+):({opt_char_class})'
    matches = re.findall(pattern1, response)
    if matches and len(matches) >= expected_count:
        matches.sort(key=lambda x: int(x[0]))
        choices = [int(m[1]) for m in matches[:expected_count]]
    else:
        pattern2 = rf'(?:答案|选项|option|answer)\s*[：:]*\s*({opt_char_class})'
        matches2 = re.findall(pattern2, response.lower())
        if len(matches2) >= expected_count:
            choices = [int(m) for m in matches2[:expected_count]]
        else:
            numbers = re.findall(rf'\b({opt_char_class})\b', response)
            if len(numbers) >= expected_count:
                choices = [int(n) for n in numbers[:expected_count]]

    choices = [c if 1 <= c <= num_options else 1 for c in choices]
    while len(choices) < expected_count:
        choices.append(1)
    return choices[:expected_count]

def ask_qwen(prompt, num_options=4):
    response_text = ''
    last_error = None

    for candidate in MODEL_CANDIDATES:
        try:
            response = client.chat.completions.create(
                model=candidate,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=128
            )
            if response.choices and response.choices[0].message:
                response_text = response.choices[0].message.content
                return parse_choices(response_text, num_options=num_options, expected_count=1)[0], candidate, response_text
        except Exception as e:
            last_error = str(e)
            continue

    return 1, 'N/A', f'All Qwen models failed: {last_error}'

In [ ]:
from tqdm.notebook import tqdm

questions_per_video = load_questions_from_csv(BENCHMARK_QA_CSV)
print('Loaded videos:', len(questions_per_video))

results = []
total_correct = 0
total_questions = 0

# 使用 tqdm 包装外层循环以显示进度条
pbar = tqdm(questions_per_video, desc="Processing videos")

for item in pbar:
    video_number = item['video_number']
    qas = item['qas']

    choices = []
    raw_responses = []
    used_models = []

    for qa_idx, qa in enumerate(qas, 1):
        single_prompt = build_single_question_prompt(qa_idx, qa)
        num_options = len(qa.get('options', []))
        if num_options <= 0:
            num_options = 4

        ans, used_model, raw_text = ask_qwen(single_prompt, num_options=num_options)
        choices.append(ans)
        used_models.append(used_model)
        raw_responses.append(raw_text)

    correct = [qa['correct_index'] for qa in qas]
    num_questions = len(qas)
    correct_count = sum([c == r for c, r in zip(choices, correct)])
    acc = correct_count / num_questions if num_questions > 0 else 0

    total_correct += correct_count
    total_questions += num_questions
    
    # 实时更新进度条描述，显示当前的整体正确率
    current_acc = total_correct / total_questions if total_questions > 0 else 0
    pbar.set_postfix({'Current Acc': f'{current_acc:.2%}'})

    results.append({
        'video_number': video_number,
        'choices': choices,
        'correct': correct,
        'accuracy': acc,
        'num_questions': num_questions,
        'correct_count': correct_count,
        'qwen_models': used_models,
        'raw_responses': raw_responses,
    })

overall_acc = total_correct / total_questions if total_questions > 0 else 0

csv_basename = Path(BENCHMARK_QA_CSV).stem
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = Path(DRIVE_OUTPUT_DIR) / f'results_{MODEL_NAME}-single_qa-{csv_basename}_{timestamp}.json'

summary = {
    'api_env_name': API_ENV_NAME,
    'model_name': MODEL_NAME,
    'overall_accuracy': overall_acc,
    'total_correct': total_correct,
    'total_questions': total_questions,
    'num_videos': len(results),
    'timestamp': datetime.now().isoformat(),
    'input_csv': BENCHMARK_QA_CSV,
    'output_dir': str(DRIVE_OUTPUT_DIR),
    'qwen_model_candidates': MODEL_CANDIDATES,
    'generation_config': {'temperature': 0.1, 'max_tokens': 128},
    'results': results,
}

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Done')
print('Overall accuracy:', f'{overall_acc:.2%}', f'({total_correct}/{total_questions})')
print('Saved to:', output_file)